# 07 - Gold: Hechos de Contratos Públicos

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

| | Tabla |
|---|---|
| Entradas | `datos_empresas.silver.de_secop_contratos`, `datos_empresas.silver.de_tasacambio`, `datos_empresas.gold.de_dim_empresas` |
| Salida | `datos_empresas.gold.de_fact_contratos` |
| Grano | Una fila por contrato (`id_contrato`) |

**Depende de `04_Silver_SECOP`, `05_Silver_TRM` y `06_Gold_Dim_Empresas`.**

## Propósito analítico

Tabla de hechos de la contratación pública, con medidas listas para agregar. Responde:

* ¿Qué entidades, sectores y departamentos concentran más valor contratado, en pesos y en dólares?
* ¿Qué proporción se adjudica por **contratación directa** y cuánto va a **pymes**?
* ¿Qué tan ejecutados están los contratos (`porcentaje_pagado`) y cuántos tienen **adiciones de tiempo**?
* ¿Cuántos proveedores están registrados en el RUES?

## Transformaciones

| Columna Gold | Cálculo | Justificación |
|---|---|---|
| `trm_fecha_firma` | TRM de Silver cuyo rango `[desde, hasta]` contiene `fecha_firma` | La TRM de fin de semana vale desde el viernes hasta el lunes, por eso el cruce es por rango y no por igualdad. Si hubiera varias coincidencias se toma la de `desde` más reciente, para que nunca se duplique un contrato. |
| `valor_contrato_usd` | `valor_contrato / trm_fecha_firma` | Permite comparar contratos de distintas fechas en una moneda estable. Queda `NULL` si la firma está fuera del periodo cubierto por la TRM. |
| `es_contratacion_directa` | `modalidad_contratacion` empieza por "Contratación directa" | Agrupa las dos variantes que trae la fuente ("Contratación directa" y "Contratación Directa (con ofertas)"). |
| `porcentaje_pagado` | `valor_pagado / valor_contrato × 100` | Indicador de ejecución financiera. Silver ya garantiza que está entre 0 y 100. |
| `duracion_dias`, `tiene_adicion_tiempo` | `fecha_fin - fecha_inicio`, `dias_adicionados > 0` | Duración y prórrogas del contrato. |
| `llave_empresa_proveedor`, `proveedor_registrado_rues` | Documento del proveedor = NIT o identificación de una empresa de `gold.de_dim_empresas` | Enlaza el hecho con la dimensión (modelo estrella). Si el mismo documento aparece en varias cámaras, se toma la menor `llave_empresa` para que el resultado sea determinista. |
| `anio_mes_firma` | Primer día del mes de firma | Llave de tiempo para cruzar con `gold.de_agg_indicadores_mes`. |

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: TRM vigente en la fecha de firma de cada contrato

In [ ]:
%python
contratos = spark.table(SILVER_SECOP)
trm = spark.table(SILVER_TRM).select("fecha_vigencia_desde", "fecha_vigencia_hasta", "valor_trm")

w_trm = Window.partitionBy("id_contrato").orderBy(F.col("fecha_vigencia_desde").desc())
trm_por_contrato = (
    contratos.select("id_contrato", "fecha_firma")
    .join(F.broadcast(trm),
          (F.col("fecha_firma") >= F.col("fecha_vigencia_desde")) & (F.col("fecha_firma") <= F.col("fecha_vigencia_hasta")),
          "inner")
    .withColumn("_rn", F.row_number().over(w_trm))
    .filter("_rn = 1")
    .select("id_contrato", F.col("valor_trm").alias("trm_fecha_firma"))
)

---
## Paso 2: Proveedor registrado en RUES (enlace con la dimensión de empresas)

In [ ]:
%python
documentos_rues = (
    spark.table(GOLD_DIM_EMPRESAS)
    .select("llave_empresa", F.explode(F.array("numero_identificacion_normalizado", "nit_normalizado")).alias("documento"))
    .filter(F.col("documento").isNotNull())
    .groupBy("documento")
    .agg(F.min("llave_empresa").alias("llave_empresa_proveedor"))
)

---
## Paso 3: Construir la tabla de hechos

In [ ]:
%python
c = contratos.alias("c")

df_fact = (
    c.join(trm_por_contrato.alias("t"), "id_contrato", "left")
     .join(documentos_rues.alias("d"), F.col("c.documento_proveedor_normalizado") == F.col("d.documento"), "left")
     .select(
        "id_contrato", "c.referencia_contrato", "c.proceso_compra",
        # Entidad
        "c.nit_entidad", "c.nombre_entidad", "c.departamento", "c.ciudad", "c.orden", "c.sector", "c.rama",
        # Contrato
        "c.tipo_contrato", "c.modalidad_contratacion",
        F.lower(F.col("c.modalidad_contratacion")).like("contrataci_n directa%").alias("es_contratacion_directa"),
        "c.estado_contrato", "c.origen_recursos", "c.es_postconflicto",
        # Proveedor
        "c.tipo_documento_proveedor", "c.documento_proveedor_normalizado", "c.proveedor_adjudicado",
        "c.es_pyme", "c.es_grupo",
        "d.llave_empresa_proveedor",
        F.col("d.llave_empresa_proveedor").isNotNull().alias("proveedor_registrado_rues"),
        # Tiempo
        "c.fecha_firma",
        F.year("c.fecha_firma").alias("anio_firma"),
        F.month("c.fecha_firma").alias("mes_firma"),
        F.trunc("c.fecha_firma", "month").alias("anio_mes_firma"),
        "c.fecha_inicio", "c.fecha_fin",
        F.datediff("c.fecha_fin", "c.fecha_inicio").alias("duracion_dias"),
        "c.dias_adicionados",
        F.coalesce(F.col("c.dias_adicionados") > 0, F.lit(False)).alias("tiene_adicion_tiempo"),
        # Medidas
        "c.valor_contrato", "c.valor_pagado", "c.valor_facturado", "c.valor_pendiente_pago",
        F.round(F.col("c.valor_pagado") / F.col("c.valor_contrato") * 100, 2).cast("decimal(5,2)").alias("porcentaje_pagado"),
        "t.trm_fecha_firma",
        F.round(F.col("c.valor_contrato") / F.col("t.trm_fecha_firma"), 2).cast("decimal(22,2)").alias("valor_contrato_usd"),
        "c.url_proceso",
     )
)

COLUMNAS_NEGOCIO = list(df_fact.columns)
df_fact = df_fact.withColumn("_hash_registro", hash_columnas(COLUMNAS_NEGOCIO))

---
## Paso 4: Escritura idempotente (MERGE por `id_contrato`)

In [ ]:
%python
m_gold = escribir_merge(
    df_fact, GOLD_FACT_CONTRATOS, ["id_contrato"], capa="gold",
    comentario="Gold - Hechos de contratacion publica SECOP II: un contrato por fila, con valor en COP y USD e indicadores.",
)

n_silver, n_gold = spark.table(SILVER_SECOP).count(), spark.table(GOLD_FACT_CONTRATOS).count()
assert n_silver == n_gold, f"El hecho ({n_gold}) debe tener un contrato por cada contrato de Silver ({n_silver})"
print(f"Contratos en Silver: {n_silver:,} | Contratos en Gold: {n_gold:,}")

---
## Paso 5: Consultas de ejemplo

In [ ]:
%sql
-- Contratación directa y participación de pymes por sector
SELECT sector,
       COUNT(*) AS contratos,
       ROUND(SUM(valor_contrato) / 1e9, 1) AS valor_miles_millones_cop,
       ROUND(100 * AVG(CASE WHEN es_contratacion_directa THEN 1 ELSE 0 END), 1) AS pct_directa,
       ROUND(100 * AVG(CASE WHEN es_pyme THEN 1 ELSE 0 END), 1) AS pct_pyme
FROM datos_empresas.gold.de_fact_contratos
GROUP BY sector
ORDER BY valor_miles_millones_cop DESC
LIMIT 10;

In [ ]:
%sql
-- Cobertura de la conversión a dólares y del enlace con RUES
SELECT COUNT(*) AS contratos,
       SUM(CASE WHEN valor_contrato_usd IS NOT NULL THEN 1 ELSE 0 END) AS con_valor_usd,
       SUM(CASE WHEN proveedor_registrado_rues THEN 1 ELSE 0 END) AS proveedor_en_rues
FROM datos_empresas.gold.de_fact_contratos;

In [ ]:
%python
finalizar({"notebook": "07_Gold_Fact_Contratos", "gold": m_gold})